In [0]:
CREATE OR replace temp VIEW mpsii_tx_claims AS 
(
SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
where (patient_id in (SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM (SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31')
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2)) and fill_date between '2023-08-01' AND '2025-07-31'
);

In [0]:

-- Raw data from the temp view for the 49 patient IDs
SELECT distinct *
FROM mpsii_tx_claims
WHERE patient_id IN (
  '2B8E9JSR','SNHEKHZZ','HWNPCZTZ','QWZMHPTJ','BS6Z00T7','HG5089NV','R99B343J','P1JF1QND','1HNFJ12Q','QJ8CMNDE',
  'K5HNZJ43','6Q831L53','ZF8XNQ6L','VM0F6LD8','Y5W4SKGM','HRJD61CS','PHDFWCCN','ZLXV6L4G','WYB4NG8H','DR65XZXF',
  'BTEQP2RC','9KZGDZ31','M9XZMW52','JJ5H8T2H','52028LXY','PC0DTBM6','V1ST6K5P','0QMXBD93','0KPZPXKH','5VKHE13W',
  'L4PXNEW9','MB6R5PRR','905ESMNC','KJDEKVJH','9J08HQPH','ERZKNJ8N','MBCKCXPB','NV1G9BPH','ND2Y0LRD','T84CH62W',
  'R880T4GS','R6BP8FQ1','T1C668E6','6YWE9Z2W','FECLC9C3','LWDDDT5V','PME0SYMW','D8NGFVRN','MBS5HSL2'
)
ORDER BY patient_id, fill_date, table_name;


In [0]:
select distinct patient_id from mpsii_tx_claims 



### Appending New Columns (Recently Treated HCP), Primary HCP in 2 year

In [0]:
create or replace temporary view most_recently_treated_hcp as
with tx_claims as (
  select distinct * from mpsii_tx_claims 
),
latest_treating_hcp as (
  SELECT
      patient_id,
      npi,
      fill_date AS latest_tx_date
  FROM (
      SELECT
          patient_id,
          npi,
          fill_date,
          ROW_NUMBER() OVER (
              PARTITION BY patient_id
              ORDER BY 
                  CASE WHEN npi IS NOT NULL THEN 1 ELSE 2 END,  -- prioritize non-null
                  fill_date DESC,
                  npi DESC
          ) AS rn
      FROM tx_claims
  ) t
  WHERE rn = 1
    -- AND npi IS NOT NULL
),
latest_treating_hcp_codes AS (
    SELECT
        l.patient_id,
        l.npi,
        concat_ws(',', collect_set(t.CODE)) AS most_recent_tx_codes_2yr
    FROM latest_treating_hcp l
    LEFT JOIN tx_claims t
      ON  l.patient_id    = t.patient_id
      AND l.npi           = t.npi
      AND l.latest_tx_date = t.fill_date      -- only codes on the latest visit
    GROUP BY l.patient_id, l.npi
),

visit_counts as (
  select patient_id, npi, count(distinct fill_date) as visit_counts
  from tx_claims
  where npi is not null
  group by 1,2 order by 3 desc
),
latest_treating_hcp_with_visits as (
  select
      a.patient_id,
      a.npi as most_recently_treated_hcp,
      b.visit_counts as no_of_visits,
      c.most_recent_tx_codes_2yr
  from latest_treating_hcp a
  left join visit_counts b
    on a.patient_id = b.patient_id
   and a.npi       = b.npi
  left join latest_treating_hcp_codes c
    on a.patient_id = c.patient_id
   and a.npi       = c.npi
),

hcp_with_other_info as (
  select a.*, concat(b.FIRST_NAME || ' ' || b.LAST_NAME) as hcp_name, b.PRIMARY_SPECIALTY as hcp_specialty, c.hco_npi, c.hco_name, c.final_zip, c.territory, c.region
  from latest_treating_hcp_with_visits as a
  left join com_edp_prd.com_raw.kom_providers as b on a.most_recently_treated_hcp = b.NPI and b.PROVIDER_TYPE = 'INDIVIDUAL'
  left join com_edp_prd.cmpa_insights_internal_schema.hcp_hco_affiliations_master_table as c on a.most_recently_treated_hcp = c.hcp_npi  
)
select patient_id,
       most_recently_treated_hcp as most_recently_treated_hcp_2yr,
       hcp_name as most_recently_treated_hcp_name_2yr,
       no_of_visits as most_recently_treated_hcp_no_of_visits_2yr,
       most_recent_tx_codes_2yr as most_recently_treated_hcp_codes_2yr,
       hcp_specialty as most_recently_treated_hcp_specialty_2yr,
       hco_npi as most_recently_treated_hcp_hco_npi,
       hco_name as most_recently_treated_hcp_hco_name,
       final_zip as most_recently_treated_hcp_final_zip_2yr,
       territory as most_recently_treated_hcp_territory_2yr,
       region as most_recently_treated_hcp_region_2yr
from hcp_with_other_info;



In [0]:
-- select count(distinct patient_id), count(*)
-- from most_recently_treated_hcp
-- -- where most_recently_treated_hcp.most_recently_treated_hcp is not null

select * from most_recently_treated_hcp

In [0]:
create or replace temporary view primary_hcp as 
with pulling_specialities as (
  select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (select * from mpsii_tx_claims) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi
),
primary_hcp as (
  select distinct n_pats as patient_id, npi 
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM pulling_specialities
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
),
pulling_demographics as (
  select a.patient_id, a.npi as primary_hcp, concat(b.FIRST_NAME || ' ' || b.LAST_NAME) as hcp_name, b.PRIMARY_SPECIALTY as hcp_specialty, c.hco_npi, c.hco_name, c.final_zip, c.territory, c.region
  from primary_hcp as a
  left join com_edp_prd.com_raw.kom_providers as b on a.npi = b.NPI and b.PROVIDER_TYPE = 'INDIVIDUAL'
  left join com_edp_prd.cmpa_insights_internal_schema.hcp_hco_affiliations_master_table as c on a.npi = c.hcp_npi
  where a.npi is not null
),
visit_counts as (
  select patient_id, npi, count(distinct fill_date) as visit_counts
  from mpsii_tx_claims
  where npi is not null
  group by 1,2 order by 3 desc
),
final_joined as (
    select a.*, b.visit_counts
    from pulling_demographics as a
    left join visit_counts as b on a.primary_hcp = b.npi and a.patient_id = b.patient_id
)
select distinct patient_id, primary_hcp as primary_hcp_2yr, hcp_name as primary_hcp_name_2yr, visit_counts as primary_hcp_no_of_visits_2yr, hcp_specialty as primary_hcp_specialty_2yr, hco_npi as primary_hcp_hco_npi_2yr, hco_name as primary_hcp_hco_name_2yr, final_zip as primary_hcp_final_zip_2yr, territory as primary_hcp_territory_2yr, region as primary_hcp_region_2yr
from final_joined

In [0]:
select * from primary_hcp

In [0]:
create or replace table com_edp_prd.cmpa_insights_internal_schema.patient360 as
select a.PATIENT_ID, a.PATIENT_YOB, a.PATIENT_AGE, a.PATIENT_GENDER, a.patient_state, a.incidence_date, a.first_incidence_treatment_date, a.latest_claim_date, a.first_dx_hcp_5yr, a.first_dx_all_visit_count_5yr, a.first_dx_last_visit_5yr, a.first_tx_hcp_5yr, a.first_tx_all_visit_count_5yr, a.first_tx_treatment_visit_count_5yr, a.first_tx_last_visit_5yr, a.most_seen_hcp1_3yr_ranked, a.most_seen_hcp1_visit_count_3yr, a.most_seen_hcp1_last_visit_3yr, a.most_seen_hcp2_3yr_ranked, a.most_seen_hcp2_visit_count_3yr, a.most_seen_hcp2_last_visit_3yr, a.most_seen_hcp3_3yr_ranked, a.most_seen_hcp3_visit_count_3yr, a.most_seen_hcp3_last_visit_3yr, a.most_seen_hcp4_3yr_ranked, a.most_seen_hcp4_visit_count_3yr, a.most_seen_hcp4_last_visit_3yr, a.most_seen_hcp5_3yr_ranked, a.most_seen_hcp5_visit_count_3yr, a.most_seen_hcp5_last_visit_3yr, b.* EXCEPT(patient_id), c.* except(patient_id)
from com_edp_prd.cmpa_insights_internal_schema.patient360 as a
left join most_recently_treated_hcp as b on a.PATIENT_ID = b.patient_id
left join primary_hcp as c on a.PATIENT_ID = c.patient_id

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient360

In [0]:
with t1 as (select distinct patient_id
from com_edp_prd.cmpa_insights_internal_schema.patient360
where primary_hcp_3yr is null),
t2 as (SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
where (patient_id in (SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM (SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31')
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2)) 
-- and fill_date between '2020-08-01' AND '2025-07-31'
)
select distinct patient_id
from t2
where patient_id in (select distinct patient_id from t1) and npi is not null
-- select distinct patient_id from t1

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping
where hco_primary_npi is not null

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient360